## 🔄 RAG vs Full Context Comparison Analysis

### 9. Comprehensive Performance Analysis: RAG Approaches vs Full Context

# RAG Reranking Value Analysis: Does Reranking Still Matter?

## 🎯 Executive Summary & Key Findings

**Research Question:** Does reranking still make sense in a post-RAG world?

### Main Conclusions (To be populated after analysis):

1. **Reranking Impact**: [TBD - Impact of reranking across different configurations]
2. **Optimal K-Values**: [TBD - Best retrieval counts for different scenarios]  
3. **Enhanced vs Basic RAG**: [TBD - When query enhancement matters]
4. **Cost-Performance Trade-offs**: [TBD - Efficiency recommendations]
5. **Production Guidelines**: [TBD - When to use each configuration]

---

## 📊 Dataset Overview

**Experimental Design:**
- **Total Records**: ~7,815 RAG experimental results (from consolidated parquet)
- **Data Source**: Concatenated and deduplicated parquet file from `data/processed/wikiqa/wikiqa_full_results.parquet`
- **Variables**:
  - `k`: Retrieval count [1, 5, 10, 20, 50, 100]
  - `enhanced_rag`: Basic vs Enhanced (with query rewriting) - derived from `retrieval_kind`
  - `rerank`: With vs Without reranking
  - Pre-computed context token counts from `context_token_count`
- **Evaluation**: LLM-as-a-judge with retrieval quality metrics

**Key Metrics:**
- **Accuracy**: Absolute correctness and context-grounded assessment
- **Retrieval Quality**: Precision@k, Recall@k, F1, MRR, Hit Rate
- **Answer Quality**: Completeness, context utilization
- **Similarity Scores**: Semantic relevance measures
- **Context Efficiency**: Token usage and cost analysis (pre-computed)

In [89]:
# Data Loading and Preprocessing
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import json
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import ttest_rel, f_oneway
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('default')
sns.set_palette("husl")

print("📚 Loading experimental results from parquet...")

# Load the consolidated parquet data with polars (faster) then convert to pandas
# df = pl.read_parquet('../../data/processed/wikiqa/wikiqa_full_results.parquet')
df = pl.read_parquet("../../data/processed/wikiqa/wikiqa_full_results_flattened.parquet")

# batch_2 = json.load(open('../../data/.cache/wikiqa/batch2_results.json'))
df_b2 = pl.read_parquet('../../batch2_results_converted.parquet').with_columns(pl.col('document_url').list.first())
initial_work = pl.concat([df, df_b2], how='diagonal_relaxed').drop('stage_2_output')
print(f"✅ Loaded {len(initial_work)} experimental records from parquet")

# KEEP all approaches including full_context for comprehensive comparison
print(f"📊 Dataset includes ALL approaches: {initial_work['retrieval_kind'].value_counts().to_dict()}")
print(f"🔧 Converting to structured DataFrame...")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
📚 Loading experimental results from parquet...
✅ Loaded 14048 experimental records from parquet
📊 Dataset includes ALL approaches: {'retrieval_kind': shape: (3,)
Series: 'retrieval_kind' [str]
[
	"full_context"
	"basic_rag"
	"enhanced_rag"
], 'count': shape: (3,)
Series: 'count' [u32]
[
	3402
	5630
	5016
]}
🔧 Converting to structured DataFrame...


In [90]:
#   1. File Discovery:
from pathlib import Path
import polars as pl

from context_is_king import DATA_DIR

cache_path = Path("../../data/.cache/wikiqa")
print(cache_path.resolve())
parquet_files = list(cache_path.glob('*.parquet'))
for file in parquet_files:
    df = pl.scan_parquet(file)
    records = df.select(pl.len()).collect().item()
    print(f"{file.name}: {records:,} records")
all_dfs = []
for file in parquet_files:
    df = pl.read_parquet(file)
    if df.select(pl.len()).item() > 10:
        df = df.with_columns(pl.lit(file.name).alias("source_file")).drop('stage_2_output') #.with_columns(pl.col('stage_2_output').cast(pl.String))
        if 'list' in str(df['document_url'].dtype).lower():
            df = df.with_columns(pl.col('document_url').list.first())
        all_dfs.append(df)


all_dfs.append(pl.read_parquet(DATA_DIR / '**/total_experiment_output.parquet'))

consolidated_df = pl.concat(all_dfs, how="diagonal")
print(f'Consolidated_df has {len(consolidated_df)} rows. ')
# Check retrieval metrics
retrieval_cols = [col for col in df.columns if col.startswith('retrieval_benchmarks_')]

# Analyze by experiment type
is_full_context = 'full_context' in file.name


/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa
50_enhanced_rag_True_200.parquet: 567 records
10_basic_rag_False_200_1758012107_69f243e9.parquet: 411 records
50_basic_rag_True_200_1758055571_2a7ffa1e.parquet: 567 records
5_enhanced_rag_False_2_1758376312_f6658091.parquet: 2 records
20_enhanced_rag_False_200_1758048508_9fb41597.parquet: 565 records
5_enhanced_rag_True_200.parquet: 567 records
1_enhanced_rag_False_200_1758021635_a2ca1adc.parquet: 609 records
1_full_context_False_200_1758060460_6ebf2140.parquet: 567 records
10_basic_rag_True_200_1758021635_02b6a934.parquet: 639 records
10_enhanced_rag_False_200_1758011666_16cdffde.parquet: 410 records
20_enhanced_rag_True_200_1758060720_e6af5fdc.parquet: 567 records
5_enhanced_rag_False_2_1758047394_facf41f0.parquet: 2 records
5_basic_rag_False_200_1758021635_11f53a54.parquet: 657 records
20_basic_rag_False_200.parquet: 567 records
50_enhanced_rag_False_200_1758048508_50d2976e.parquet: 565 records
1_bas

In [253]:
paths = [
    "/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/100_enhanced_rag_False_200_1758445497_1c9aee1a.parquet",
    "/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/100_enhanced_rag_True_200_1758445497_944c4f4e.parquet",
    "/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/200_basic_rag_False_200_1758445497_55e29069.parquet",
    "/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/200_basic_rag_True_200_1758445497_cb098fca.parquet",
    "/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/100_basic_rag_False_200_1758452635_575c411f.parquet",
    "/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/100_basic_rag_True_200_1758453750_8ec58f45.parquet",
    "/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/200_enhanced_rag_False_200_1758453633_ae4aad2e.parquet",
    "/Users/baukebrenninkmeijer/Developer/pydata-2025-context-is-king/data/.cache/wikiqa/200_enhanced_rag_True_200_1758445497_e365b325.parquet"
]


lazy_dfs = [pl.scan_parquet(path) for path in paths]

# Concatenate with vertical_relaxed to handle schema differences automatically
k100200 = pl.concat(lazy_dfs, how="vertical_relaxed").with_columns(pl.col('document_url').list.first()).collect()
k100200.schema

Schema([('question_index', Int64),
        ('k', Int64),
        ('retrieval_kind', String),
        ('rerank', Boolean),
        ('n_question', Int64),
        ('judge_result_absolute_assessment_is_correct', Boolean),
        ('judge_result_absolute_assessment_confidence', Float64),
        ('judge_result_absolute_assessment_reasoning', String),
        ('judge_result_context_grounded_assessment_is_correct_given_context',
         Boolean),
        ('judge_result_context_grounded_assessment_confidence', Float64),
        ('judge_result_context_grounded_assessment_reasoning', String),
        ('judge_result_answer_completeness', Float64),
        ('judge_result_context_utilization', Float64),
        ('retrieval_benchmarks_precision@1', Float64),
        ('retrieval_benchmarks_recall@1', Float64),
        ('retrieval_benchmarks_f1@1', Float64),
        ('retrieval_benchmarks_precision@5', Float64),
        ('retrieval_benchmarks_recall@5', Float64),
        ('retrieval_benchmarks_f1@5'

In [254]:
full_df_pl = pl.concat([initial_work, consolidated_df, k100200], how='diagonal_relaxed').unique()
print(len(full_df_pl))

39989


In [255]:
consolidated_df.select(pl.all().is_null().sum())

k,retrieval_kind,rerank,n_question,judge_result_absolute_assessment_is_correct,judge_result_absolute_assessment_confidence,judge_result_absolute_assessment_reasoning,judge_result_context_grounded_assessment_is_correct_given_context,judge_result_context_grounded_assessment_confidence,judge_result_context_grounded_assessment_reasoning,judge_result_answer_completeness,judge_result_context_utilization,retrieval_benchmarks_precision@1,retrieval_benchmarks_recall@1,retrieval_benchmarks_f1@1,retrieval_benchmarks_precision@5,retrieval_benchmarks_recall@5,retrieval_benchmarks_f1@5,retrieval_benchmarks_precision@10,retrieval_benchmarks_recall@10,retrieval_benchmarks_f1@10,retrieval_benchmarks_precision@20,retrieval_benchmarks_recall@20,retrieval_benchmarks_f1@20,retrieval_benchmarks_precision@50,retrieval_benchmarks_recall@50,retrieval_benchmarks_f1@50,retrieval_benchmarks_mrr,retrieval_benchmarks_hit_rate,retrieval_benchmarks_avg_similarity,retrieval_benchmarks_max_similarity,retrieval_benchmarks_min_similarity,context_token_count,generated_answer,iteration,retrieved_chunks,probability_human,generated_question,question_reasoning,main_chunk_id,answer,status,document_url,collection_suffix,source_file,question_index,judge_result,retrieval_benchmarks,stage_2_output,error_occurred,error_type,error_message,error_code,content_policy_violation,input_tokens
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,4909,4909,4909,4909,4909,4909,4909,4909,3648,3648,3648,6987,6987,6987,11686,11686,11686,16901,16901,16901,19167,19167,19167,3648,3648,3648,3648,3648,0,54,0,0,0,0,0,0,0,0,0,0,6819,2835,19081,21433,21433,20791,21433,21433,21433,20791,20791


In [256]:
df = full_df_pl.to_pandas()

In [257]:
# Convert to DataFrame - data is already flattened
# Create enhanced_rag boolean from retrieval_kind string - but handle full_context separately
df['enhanced_rag'] = df['retrieval_kind'] == 'enhanced_rag'

# Handle document_url if it's still a list (from our concatenation fix)
if df['document_url'].dtype == 'object':
    df['document_url'] = df['document_url'].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else x
    )

# Rename columns to match expected names
column_mapping = {
    'judge_result_absolute_assessment_is_correct': 'is_correct',
    'judge_result_absolute_assessment_confidence': 'confidence',
    'judge_result_context_grounded_assessment_is_correct_given_context': 'is_correct_given_context',
    'judge_result_context_grounded_assessment_confidence': 'context_confidence',
    'judge_result_answer_completeness': 'answer_completeness',
    'judge_result_context_utilization': 'context_utilization',
    'retrieval_benchmarks_precision@1': 'precision_at_1',
    'retrieval_benchmarks_recall@1': 'recall_at_1',
    'retrieval_benchmarks_f1@1': 'f1_at_1',
    'retrieval_benchmarks_mrr': 'mrr',
    'retrieval_benchmarks_hit_rate': 'hit_rate',
    'retrieval_benchmarks_avg_similarity': 'avg_similarity',
    'retrieval_benchmarks_max_similarity': 'max_similarity',
    'retrieval_benchmarks_min_similarity': 'min_similarity',
}

df = df.rename(columns=column_mapping)

print(f"📊 DataFrame shape: {df.shape}")
print(f"🔍 Retrieval kinds: {df['retrieval_kind'].value_counts().to_dict()}")
print(f"📈 Enhanced RAG boolean: {df['enhanced_rag'].value_counts().to_dict()}")
print(f"📈 Data types:")
print(df.dtypes)

📊 DataFrame shape: (39989, 60)
🔍 Retrieval kinds: {'enhanced_rag': 16511, 'basic_rag': 16428, 'full_context': 7050}
📈 Enhanced RAG boolean: {False: 23478, True: 16511}
📈 Data types:
k                                                       int64
retrieval_kind                                         object
rerank                                                   bool
n_question                                              int64
is_correct                                             object
confidence                                            float64
judge_result_absolute_assessment_reasoning             object
is_correct_given_context                               object
context_confidence                                    float64
judge_result_context_grounded_assessment_reasoning     object
answer_completeness                                   float64
context_utilization                                   float64
precision_at_1                                        float64
recall_at_1 

In [258]:
# Data Quality Check and Basic Statistics
print("🔍 DATA QUALITY OVERVIEW")
print("=" * 50)


# Create summary variables for analysis - 5 distinct approaches
def create_config_label(row):
    """Create config labels for 5 distinct approaches"""
    if row["retrieval_kind"] == "full_context":
        return "Full Context"
    elif row["retrieval_kind"] == "enhanced_rag":
        if row["rerank"]:
            return "Enhanced RAG + Rerank"
        else:
            return "Enhanced RAG"
    else:  # basic_rag
        if row["rerank"]:
            return "Basic RAG + Rerank"
        else:
            return "Basic RAG"


df["config_label"] = df.apply(create_config_label, axis=1)

# Also create cleaner category variables
df["approach_type"] = df["retrieval_kind"].map(
    {"full_context": "Full Context", "enhanced_rag": "Enhanced RAG", "basic_rag": "Basic RAG"}
)

df["rag_type"] = df.apply(
    lambda x: "Full Context"
    if x["retrieval_kind"] == "full_context"
    else ("Enhanced" if x["enhanced_rag"] else "Basic"),
    axis=1,
)

df["rerank_type"] = df["rerank"].map({True: "With Reranking", False: "No Reranking"})

# Check configuration distribution
print("\n📊 Configuration Distribution by Retrieval Kind and Reranking:")
config_summary = df.groupby(["retrieval_kind", "rerank"]).size().reset_index(name="count")
print(config_summary.to_string(index=False))

print(f"\n📈 5 Distinct Approach Configurations:")
config_counts = df["config_label"].value_counts()
print(config_counts)

print(f"\n📊 Expected 5 Categories:")
print("1. Basic RAG")
print("2. Basic RAG + Rerank")
print("3. Enhanced RAG")
print("4. Enhanced RAG + Rerank")
print("5. Full Context")

# Verify we have all 5 categories
expected_configs = ["Basic RAG", "Basic RAG + Rerank", "Enhanced RAG", "Enhanced RAG + Rerank", "Full Context"]
missing_configs = [config for config in expected_configs if config not in config_counts.index]
if missing_configs:
    print(f"\n⚠️  Missing configurations: {missing_configs}")
else:
    print(f"\n✅ All 5 expected configurations present!")

# Check missing values
print(f"\n❌ Missing Values (top 10):")
missing_summary = df.isnull().sum()
missing_data = missing_summary[missing_summary > 0]
if len(missing_data) > 0:
    print(missing_data.head(10))
else:
    print("No missing values in key columns")

# Basic performance statistics
print(f"\n📈 Key Performance Metrics:")
performance_cols = [
    "is_correct",
    "is_correct_given_context",
    "answer_completeness",
    "context_utilization",
    "precision_at_1",
    "recall_at_1",
    "f1_at_1",
    "mrr",
    "hit_rate",
    "avg_similarity",
]

available_perf_cols = [col for col in performance_cols if col in df.columns]
if available_perf_cols:
    perf_summary = df[available_perf_cols].describe()
    print(perf_summary.round(3))

print(f"\n✅ Data preprocessing complete! 5 distinct approaches properly separated.")

🔍 DATA QUALITY OVERVIEW

📊 Configuration Distribution by Retrieval Kind and Reranking:
retrieval_kind  rerank  count
     basic_rag   False   8134
     basic_rag    True   8294
  enhanced_rag   False   8290
  enhanced_rag    True   8221
  full_context   False   7050

📈 5 Distinct Approach Configurations:
config_label
Basic RAG + Rerank       8294
Enhanced RAG             8290
Enhanced RAG + Rerank    8221
Basic RAG                8134
Full Context             7050
Name: count, dtype: int64

📊 Expected 5 Categories:
1. Basic RAG
2. Basic RAG + Rerank
3. Enhanced RAG
4. Enhanced RAG + Rerank
5. Full Context

✅ All 5 expected configurations present!

❌ Missing Values (top 10):
is_correct                                            10927
confidence                                            10927
judge_result_absolute_assessment_reasoning            10927
is_correct_given_context                              10927
context_confidence                                    10927
judge_result_cont

In [259]:
# Performance Comparison: Full Context vs RAG Approaches
def compare_approaches(df, metrics):
    """Compare performance across all approaches"""

    comparison_results = {}

    for metric in metrics:
        if metric not in df.columns:
            print(f"⚠️  Metric {metric} not available")
            continue

        # Overall approach comparison
        approach_stats = df.groupby("retrieval_kind")[metric].agg(["count", "mean", "std", "min", "max"]).round(4)

        # Detailed approach comparison (including reranking)
        detailed_stats = df.groupby("config_label")[metric].agg(["count", "mean", "std", "min", "max"]).round(4)

        comparison_results[metric] = {"approach_stats": approach_stats, "detailed_stats": detailed_stats}

        print(f"\n📊 {metric.replace('_', ' ').title()} - Overall Approach Comparison:")
        print(approach_stats.to_string())

        print(f"\n📊 {metric.replace('_', ' ').title()} - Detailed Approach Comparison:")
        print(detailed_stats.to_string())

        # Calculate improvement/degradation from Full Context
        if "Full Context" in approach_stats.index:
            full_context_mean = approach_stats.loc["Full Context", "mean"]

            print(f"\n📈 Performance vs Full Context ({full_context_mean:.4f}):")
            for approach in approach_stats.index:
                if approach != "Full Context":
                    approach_mean = approach_stats.loc[approach, "mean"]
                    diff = approach_mean - full_context_mean
                    diff_pct = (diff / full_context_mean * 100) if full_context_mean != 0 else 0
                    status = "📈" if diff > 0 else "📉" if diff < 0 else "➡️"
                    print(f"  {status} {approach}: {diff:+.4f} ({diff_pct:+.2f}%)")

    return comparison_results


# Run comprehensive comparison
key_metrics = [
    "is_correct",
    "is_correct_given_context",
    "answer_completeness",
    "context_utilization",
    "f1_at_1",
    "retrieval_benchmarks_recall@5",
    "mrr",
    "hit_rate",
    "context_token_count",
]

approach_comparison_results = compare_approaches(df, key_metrics)


📊 Is Correct - Overall Approach Comparison:
                count      mean     std    min   max
retrieval_kind                                      
basic_rag       10502  0.800324  0.3998  False  True
enhanced_rag    12077  0.823135  0.3816  False  True
full_context     6483  0.921641  0.2688  False  True

📊 Is Correct - Detailed Approach Comparison:
                       count      mean     std    min   max
config_label                                               
Basic RAG               4395  0.797497  0.4019  False  True
Basic RAG + Rerank      6107  0.802358  0.3983  False  True
Enhanced RAG            6567  0.821684  0.3828  False  True
Enhanced RAG + Rerank   5510  0.824864  0.3801  False  True
Full Context            6483  0.921641  0.2688  False  True

📊 Is Correct Given Context - Overall Approach Comparison:
                count      mean     std    min   max
retrieval_kind                                      
basic_rag       10502  0.983908  0.1258  False  True
enhanc

In [260]:
# RAG vs Full Context Comprehensive Analysis
print("🔄 RAG vs FULL CONTEXT COMPREHENSIVE ANALYSIS")
print("=" * 60)

# Check what approaches we have in the data
print("\n📊 Available Approaches in Dataset:")
approach_counts = df['retrieval_kind'].value_counts()
print(approach_counts)

# Create approach categories for comparison
df['approach_category'] = df['retrieval_kind'].map({
    'full_context': 'Full Context',
    'enhanced_rag': 'Enhanced RAG',
    'basic_rag': 'Basic RAG'
})

# Add reranking information to approach labels
df['detailed_approach'] = df.apply(lambda x: 
    f"{x['approach_category']} {'+ Rerank' if x['rerank'] and x['approach_category'] != 'Full Context' else ''}".strip(), 
    axis=1
)

print(f"\n📈 Approach Distribution:")
detailed_counts = df['detailed_approach'].value_counts()
print(detailed_counts)

🔄 RAG vs FULL CONTEXT COMPREHENSIVE ANALYSIS

📊 Available Approaches in Dataset:
retrieval_kind
enhanced_rag    16511
basic_rag       16428
full_context     7050
Name: count, dtype: int64

📈 Approach Distribution:
detailed_approach
Basic RAG + Rerank       8294
Enhanced RAG             8290
Enhanced RAG + Rerank    8221
Basic RAG                8134
Full Context             7050
Name: count, dtype: int64


In [261]:
# # Data Quality Check and Basic Statistics
# print("🔍 DATA QUALITY OVERVIEW")
# print("=" * 50)

# # Create summary variables for analysis
# df['config_label'] = df.apply(lambda x: f"{'Enhanced' if x['enhanced_rag'] else 'Basic'} RAG {'+ Rerank' if x['rerank'] else ''}", axis=1)
# # df.loc[df.retrieval_kind == 'full_context', 'config_label'] = ''
# df['rag_type'] = df['enhanced_rag'].map({True: 'Enhanced', False: 'Basic'})
# df['rerank_type'] = df['rerank'].map({True: 'With Reranking', False: 'No Reranking'})

# # Check configuration distribution
# print("\n📊 Configuration Distribution:")
# config_summary = df.groupby(['k', 'enhanced_rag', 'rerank']).size().reset_index(name='count')
# print(config_summary.to_string(index=False))

# # Check missing values
# print(f"\n❌ Missing Values:")
# missing_summary = df.isnull().sum()
# missing_data = missing_summary[missing_summary > 0]
# if len(missing_data) > 0:
#     print(missing_data)
# else:
#     print("No missing values in key columns")

# # Basic performance statistics
# print(f"\n📈 Key Performance Metrics:")
# performance_cols = ['is_correct', 'is_correct_given_context', 'answer_completeness', 
#                    'context_utilization', 'precision_at_1', 'recall_at_1', 'f1_at_1', 
#                    'mrr', 'hit_rate', 'avg_similarity']

# available_perf_cols = [col for col in performance_cols if col in df.columns]
# perf_summary = df[available_perf_cols].describe()
# print(perf_summary.round(3))

# print(f"\n✅ Data preprocessing complete!")

In [262]:
df.config_label.value_counts()

config_label
Basic RAG + Rerank       8294
Enhanced RAG             8290
Enhanced RAG + Rerank    8221
Basic RAG                8134
Full Context             7050
Name: count, dtype: int64

In [263]:
# Using Pre-computed Context Token Counts
print("🔢 USING PRE-COMPUTED CONTEXT TOKEN COUNTS")
print("=" * 50)

# Use existing context_token_count column from parquet
df['context_tokens_used'] = df['context_token_count']

# Calculate tokens per k
df['tokens_per_k'] = df['context_tokens_used'] / df['k'].replace(0, 1)

# Summary statistics
print(f"\n📊 Context Token Statistics:")
token_stats = df[['context_tokens_used', 'tokens_per_k']].describe()
print(token_stats.round(1))

# Check for any records with zero tokens
zero_tokens = df[df['context_tokens_used'] == 0]
if len(zero_tokens) > 0:
    print(f"\n⚠️  Found {len(zero_tokens)} records with zero context tokens")

print(f"\n📈 Context Tokens by Configuration:")
token_by_config = (
    df.groupby(["k", "rag_type", "rerank_type"])["context_tokens_used"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(1)
)
print(token_by_config)

print(f"\n✅ Context token data loaded from parquet!")
print(f"📝 Using columns: context_tokens_used, tokens_per_k")

🔢 USING PRE-COMPUTED CONTEXT TOKEN COUNTS

📊 Context Token Statistics:
       context_tokens_used  tokens_per_k
count              39989.0       39989.0
mean               14484.2        2181.6
std                17148.7        6741.0
min                   11.0          11.0
25%                 1935.0         305.2
50%                 6568.0         385.5
75%                20685.0         476.8
max                88057.0       59323.0

📈 Context Tokens by Configuration:
                                 count     mean      std    min    max
k   rag_type     rerank_type                                          
1   Basic        No Reranking     1707    350.1    121.1     11    596
                 With Reranking   1608    349.6    122.4     11    608
    Enhanced     No Reranking     1218    346.3    121.2     11    596
                 With Reranking   1216    345.9    120.8     11    596
    Full Context No Reranking     2268  27974.0   8817.4  19571  59323
5   Basic        No Reranki

In [264]:
from pathlib import Path
import chromadb
from chromadb import Collection

CHROMA_PATH = Path("../../chroma_wikitext")

chroma_client = chromadb.PersistentClient(path=str(CHROMA_PATH))
collections = chroma_client.list_collections()

In [265]:
print("📚 COLLECTION SIZE ANALYSIS")
print("=" * 50)

df = df.drop(columns=["total_chunks_referenced", "total_chunks_in_collection"], errors="ignore")

# Analyze collection suffix (datasource) information
if "collection_suffix" in df.columns:
    # Get collection size statistics
    collection_stats = (
        df.groupby(["collection_suffix", "document_url"])
        .agg(
            {
                "k": "count",  # Number of experiments per document
                "main_chunk_id": "nunique",  # Unique chunk IDs per document
            }
        )
        .reset_index()
    )

    all_collections = chroma_client.list_collections()
    available_collection_names = [col.name for col in all_collections]
    collection_counts = []
    for idx, row in collection_stats.iterrows():
        collection_name = row["collection_suffix"]
        collection_name = f"wikiqa_{collection_name}"
        if collection_name not in available_collection_names:
            continue
        try:
            collection: Collection = chroma_client.get_collection(name=collection_name)
            total_chunks = collection.count()
            collection_counts.append(total_chunks)
            # collection_stats.at[idx, 'total_chunks_in_collection'] = total_chunks
        except Exception as e:
            collection_counts.append(np.nan)
            # collection_stats.at[idx, 'total_chunks_in_collection'] = np.nan
            print(f"⚠️  Could not retrieve collection '{collection_name}': {e}")
    collection_stats["total_chunks_in_collection"] = collection_counts

    # Rename columns for clarity
    collection_stats = collection_stats.rename(
        columns={"k": "experiment_count", "main_chunk_id": "unique_chunks_referenced"}
    )

    # Get total chunks per collection (datasource)
    collection_summary = (
        collection_stats.groupby("collection_suffix")
        .agg(
            {
                "document_url": "nunique",  # Number of unique documents
                "experiment_count": "sum",  # Total experiments
                "unique_chunks_referenced": "sum",  # Total unique chunks referenced
                "total_chunks_in_collection": "first",
            }
        )
        .reset_index()
    )

    collection_summary = collection_summary.rename(
        columns={
            "document_url": "unique_documents",
            "experiment_count": "total_experiments",
            "unique_chunks_referenced": "total_chunks_referenced",
        }
    )

    print(f"\n📊 Collection Summary (by datasource):")
    print(collection_summary.to_string(index=False))

    # Document-level analysis
    print(f"\n📄 Document-level Analysis:")
    doc_analysis = (
        collection_stats.groupby("document_url")
        .agg({"collection_suffix": "first", "experiment_count": "sum", "unique_chunks_referenced": "sum"})
        .sort_values("unique_chunks_referenced", ascending=False)
    )

    print(f"Top 10 documents by chunk count:")
    print(doc_analysis.head(10).to_string())

    # Collection size distribution
    print(f"\n📈 Collection Size Distribution:")
    size_distribution = collection_summary["total_chunks_referenced"].describe()
    print(size_distribution.round(1))

    # Add collection info to main dataframe for further analysis
    df = df.merge(
        collection_summary[["collection_suffix", "total_chunks_referenced", "total_chunks_in_collection"]],
        on="collection_suffix",
        how="left",
    )

    print(f"\n✅ Added 'total_chunks_referenced' column to main dataframe")
    print(f"📝 This represents the total number of chunks available in each document's collection")

else:
    print("⚠️  'collection_suffix' column not available - cannot analyze collection sizes")

# Analyze relationship between collection size and retrieval performance
if "total_chunks_referenced" in df.columns and "hit_rate" in df.columns:
    print(f"\n🎯 Collection Size vs Retrieval Performance:")

    # Correlation analysis
    valid_data = df[["total_chunks_referenced", "hit_rate", "precision_at_1", "recall_at_1"]].dropna()

    if len(valid_data) > 0:
        correlations = valid_data.corr()["total_chunks_referenced"].drop("total_chunks_referenced")
        print(f"Correlations with collection size:")
        for metric, corr in correlations.items():
            print(f"  {metric}: {corr:.3f}")

        # Group by collection size ranges
        df["collection_size_range"] = pd.cut(
            df["total_chunks_in_collection"],
            bins=[0, 100, 500, 1000, 5000, float("inf")],
            labels=["Very Small (<100)", "Small (100-500)", "Medium (500-1K)", "Large (1K-5K)", "Very Large (5K+)"],
        )

        size_performance = (
            df.groupby("collection_size_range")[["hit_rate", "precision_at_1", "recall_at_1"]].mean().round(3)
        )
        print(f"\nPerformance by collection size range:")
        print(size_performance.to_string())
    else:
        print("⚠️  Not enough valid data for correlation analysis")

print(f"\n✅ Collection size analysis complete!")

📚 COLLECTION SIZE ANALYSIS

📊 Collection Summary (by datasource):
                        collection_suffix  unique_documents  total_experiments  total_chunks_referenced  total_chunks_in_collection
          History_of_the_Byzantine_Empire                 1               6549                       38                         115
List_of_smoking_bans_in_the_United_States                 1                933                       32                         339
                             Muhammad_Ali                 1               6423                       33                         114
                               Sound_film                 1               7196                       34                         106
                       The_Flash_season_4                 1              18888                       82                         255

📄 Document-level Analysis:
Top 10 documents by chunk count:
                                                                                 

In [266]:
df['total_chunks_in_collection'].mean()

np.float64(184.57225737077695)

## 🧮 Core Performance Analysis

### 1. Performance by Configuration Matrix

Let's analyze performance across all experimental configurations: Enhanced/Basic RAG × With/Without Reranking × K-values

In [225]:
df.config_label.value_counts()

config_label
Basic RAG                15184
Basic RAG + Rerank        8294
Enhanced RAG              8290
Enhanced RAG + Rerank     7654
Name: count, dtype: int64

In [206]:
# Performance Analysis by Configuration
def analyze_by_configuration(df, metric_col):
    """Analyze performance metric by all experimental configurations"""
    if metric_col not in df.columns:
        print(f"⚠️  Metric {metric_col} not available")
        return None

    # Group by all configuration variables
    analysis = (
        df.groupby(["k", "rag_type", "rerank_type"])[metric_col].agg(["count", "mean", "std", "min", "max"]).round(4)
    )

    return analysis


# Key metrics to analyze
metrics_to_analyze = [
    ("is_correct", "Accuracy (Absolute)"),
    ("is_correct_given_context", "Accuracy (Context-Grounded)"),
    ("answer_completeness", "Answer Completeness"),
    ("context_utilization", "Context Utilization"),
    ("precision_at_1", "Precision@1"),
    ("recall_at_1", "Recall@1"),
    ("f1_at_1", "F1@1"),
    ("mrr", "Mean Reciprocal Rank"),
    ("hit_rate", "Hit Rate"),
    ("avg_similarity", "Average Similarity"),
    ("context_tokens_used", "Context Tokens Used"),
    ("tokens_per_k", "Tokens per Retrieved Document"),
]

print("🎯 PERFORMANCE BY CONFIGURATION")
print("=" * 60)

performance_results = {}
for metric_col, metric_name in metrics_to_analyze:
    if metric_col in df.columns and df[metric_col].notna().any():
        print(f"\n📊 {metric_name}:")
        result = analyze_by_configuration(df, metric_col)
        if result is not None:
            print(result.to_string())
            performance_results[metric_col] = result
    else:
        print(f"⚠️  {metric_name} not available or all null")

🎯 PERFORMANCE BY CONFIGURATION

📊 Accuracy (Absolute):
                             count      mean     std    min   max
k   rag_type rerank_type                                         
1   Basic    No Reranking     2270  0.845815  0.3612  False  True
             With Reranking    984  0.580285  0.4938  False  True
    Enhanced No Reranking      609  0.610837  0.4880  False  True
             With Reranking    566  0.600707  0.4902  False  True
5   Basic    No Reranking     2079  0.816258  0.3874  False  True
             With Reranking    798  0.670426  0.4704  False  True
    Enhanced No Reranking     1687  0.721399  0.4484  False  True
             With Reranking    807  0.739777  0.4390  False  True
10  Basic    No Reranking     1761  0.852924  0.3543  False  True
             With Reranking    639  0.810642  0.3921  False  True
    Enhanced No Reranking      410  0.795122  0.4041  False  True
             With Reranking   1302  0.847158  0.3600  False  True
20  Basic    No Reran

## 📈 Comprehensive Visualization Suite

### 2. Retrieval Performance Grid: Enhanced/Basic × Rerank/No-Rerank

In [269]:
import sys
sys.path.append('../..')
from context_is_king import IMG_DIR
from updated_plotting_functions import plot_retrieval_grid_with_full_context
for metric in key_metrics + ['retrieval_benchmarks_recall@5']:
    if metric in df.columns:
        fig = plot_retrieval_grid_with_full_context(df, metric)
        if fig:
            fig.write_image(IMG_DIR / f'reranking_experiment_retrieval_performance_grid_{metric}.svg', height=500, width=900)
            fig.show()
    else:
        print(f"⚠️  Skipping {metric} - not available")

### 3. Performance Heatmap: Configuration × K-values

In [50]:
df.is_correct.sum() / len(df)

0.6971359361641494

In [251]:
df.groupby(['config_label', 'k']).size().reset_index()

,config_label,k,0
0,Basic RAG,1,1707
1,Basic RAG,5,1602
2,Basic RAG,10,1449
3,Basic RAG,20,1110
4,Basic RAG,50,1134
5,Basic RAG,100,566
6,Basic RAG,200,566
7,Basic RAG + Rerank,1,1608
8,Basic RAG + Rerank,5,1722
9,Basic RAG + Rerank,10,1278


In [ ]:
'Full Context' in df.config_label 

False

In [271]:
df.config_label.value_counts()

config_label
Basic RAG + Rerank       8294
Enhanced RAG             8290
Enhanced RAG + Rerank    8221
Basic RAG                8134
Full Context             7050
Name: count, dtype: int64

In [275]:
from context_is_king.plotting.reranking import create_performance_heatmap

# Use it exactly like before, but without errors
fig = create_performance_heatmap(df, metric="is_correct")
if fig:
    fig.write_image(IMG_DIR / 'heatmap_is_correct.svg')
    fig.show()

## ✅ **CORRECTED: 4 RAG Approaches vs Full Context Analysis**

**The analysis below properly separates the 5 distinct approaches:**

1. **Basic RAG** (no reranking)
2. **Basic RAG + Rerank** 
3. **Enhanced RAG** (no reranking)
4. **Enhanced RAG + Rerank**
5. **Full Context** (baseline)

This provides a comprehensive comparison of all RAG variants against the Full Context baseline.

In [172]:
# Final Summary: RAG vs Full Context Key Insights
print("\n🎯 RAG vs FULL CONTEXT: KEY INSIGHTS SUMMARY")
print("=" * 60)

# 1. Performance Summary
print("\n1. PERFORMANCE COMPARISON SUMMARY")
print("-" * 40)

if "approach_comparison_results" in locals():
    for metric in ["is_correct", "f1_at_1", "mrr", "context_tokens_used"]:
        if metric in approach_comparison_results:
            approach_stats = approach_comparison_results[metric]["approach_stats"]

            if "Full Context" in approach_stats.index:
                full_context_score = approach_stats.loc["Full Context", "mean"]
                print(f"\n📊 {metric.replace('_', ' ').title()}:")
                print(f"  Full Context: {full_context_score:.4f}")

                for approach in approach_stats.index:
                    if approach != "Full Context":
                        approach_score = approach_stats.loc[approach, "mean"]
                        diff = approach_score - full_context_score
                        diff_pct = (diff / full_context_score * 100) if full_context_score != 0 else 0
                        print(f"  {approach}: {approach_score:.4f} ({diff_pct:+.2f}%)")

# 2. Statistical Significance Summary
print("\n\n2. STATISTICAL SIGNIFICANCE SUMMARY")
print("-" * 40)

if "significance_vs_full_context" in locals():
    for metric, results in significance_vs_full_context.items():
        if results is not None and len(results) > 0:
            sig_results = results[results["significant"] == True]
            print(
                f"\n{metric}: {len(sig_results)}/{len(results)} RAG approaches show significant difference from Full Context"
            )

            if len(sig_results) > 0:
                for _, row in sig_results.iterrows():
                    direction = "better" if row["difference"] > 0 else "worse"
                    print(f"  • {row['approach']}: {row['difference_pct']:+.2f}% ({direction}, p={row['p_value']:.4f})")

# 3. Efficiency and Cost Analysis
if "context_tokens_used" in df.columns:
    print("\n\n3. EFFICIENCY & COST ANALYSIS")
    print("-" * 40)

    token_comparison = df.groupby("approach_category")["context_tokens_used"].mean()

    if "Full Context" in token_comparison.index:
        full_context_tokens = token_comparison["Full Context"]
        print(f"\n💾 Token Usage vs Full Context ({full_context_tokens:.0f} tokens):")

        for approach, tokens in token_comparison.items():
            if approach != "Full Context":
                reduction_pct = (full_context_tokens - tokens) / full_context_tokens * 100
                print(f"  {approach}: {tokens:.0f} tokens ({reduction_pct:+.1f}% vs Full Context)")

# 4. Recommendations
print("\n\n4. PRODUCTION RECOMMENDATIONS")
print("-" * 40)

print("\n🎯 Key Recommendations:")
print("\n• Performance: Based on the analysis above...")
print("• Efficiency: RAG approaches show token usage benefits...")
print("• Cost-Effectiveness: Consider the trade-offs between...")
print("• Use Cases: Full Context when... vs RAG when...")

print("\n✅ Full Context vs RAG analysis complete!")
print("\n📝 Update the executive summary with these insights!")


🎯 RAG vs FULL CONTEXT: KEY INSIGHTS SUMMARY

1. PERFORMANCE COMPARISON SUMMARY
----------------------------------------


2. STATISTICAL SIGNIFICANCE SUMMARY
----------------------------------------


3. EFFICIENCY & COST ANALYSIS
----------------------------------------

💾 Token Usage vs Full Context (27681 tokens):
  Basic RAG: 5557 tokens (+79.9% vs Full Context)
  Enhanced RAG: 6486 tokens (+76.6% vs Full Context)


4. PRODUCTION RECOMMENDATIONS
----------------------------------------

🎯 Key Recommendations:

• Performance: Based on the analysis above...
• Efficiency: RAG approaches show token usage benefits...
• Cost-Effectiveness: Consider the trade-offs between...
• Use Cases: Full Context when... vs RAG when...

✅ Full Context vs RAG analysis complete!

📝 Update the executive summary with these insights!


### 4. Reranking Impact Analysis

In [277]:
# Reranking Impact Analysis
def analyze_reranking_impact(df, metric="is_correct"):
    """Calculate the performance delta from adding reranking"""

    if metric not in df.columns:
        print(f"⚠️  Metric {metric} not available")
        return None

    # Filter valid data
    analysis_df = df.dropna(subset=[metric])

    # Calculate mean performance for with/without reranking
    rerank_comparison = analysis_df.groupby(["k", "rag_type", "rerank_type"])[metric].mean().unstack()

    if "With Reranking" not in rerank_comparison.columns or "No Reranking" not in rerank_comparison.columns:
        print(f"⚠️  Missing reranking comparison data for {metric}")
        return None

    # Calculate improvement from reranking
    rerank_comparison["rerank_improvement"] = rerank_comparison["With Reranking"] - rerank_comparison["No Reranking"]
    rerank_comparison["rerank_improvement_pct"] = (
        rerank_comparison["rerank_improvement"] / rerank_comparison["No Reranking"] * 100
    )

    return rerank_comparison


# Analyze reranking impact for key metrics
print("🎯 RERANKING IMPACT ANALYSIS")
print("=" * 50)

reranking_results = {}
impact_metrics = ["is_correct", "is_correct_given_context", "f1_at_1", "mrr", "hit_rate"]

for metric in impact_metrics:
    if metric in df.columns:
        print(f"\n📊 {metric.replace('_', ' ').title()} Reranking Impact:")
        result = analyze_reranking_impact(df, metric)
        if result is not None:
            print(result.round(4).to_string())
            reranking_results[metric] = result
    else:
        print(f"⚠️  {metric} not available")

🎯 RERANKING IMPACT ANALYSIS

📊 Is Correct Reranking Impact:
rerank_type      No Reranking With Reranking rerank_improvement rerank_improvement_pct
k   rag_type                                                                          
1   Basic            0.606327       0.580285          -0.026042              -4.295098
    Enhanced         0.610837       0.600707          -0.010131              -1.658498
    Full Context     0.925926            NaN                NaN                    NaN
5   Basic            0.691005       0.670426          -0.020579              -2.978157
    Enhanced         0.721399       0.739777           0.018378               2.547553
    Full Context     0.920635            NaN                NaN                    NaN
10  Basic            0.736842       0.810642             0.0738              10.015649
    Enhanced         0.795122       0.847158           0.052036               6.544438
    Full Context     0.917108            NaN                NaN       

In [278]:
# Visualization of Reranking Impact
def plot_reranking_delta(df, metric="is_correct"):
    """Plot the improvement from reranking across k-values and RAG types"""

    if metric not in df.columns:
        return None
    
    df = df.copy()

    df[metric] = pd.to_numeric(df[metric], errors='coerce')

    analysis_df = df.dropna(subset=[metric])[(df.retrieval_kind != 'full_context') & (df.k > 1)]

    # Get mean performance by configuration
    pivot_with = analysis_df[analysis_df["rerank"] == True].groupby(["k", "rag_type"])[metric].mean()
    pivot_without = analysis_df[analysis_df["rerank"] == False].groupby(["k", "rag_type"])[metric].mean()

    # Calculate improvement
    improvement_df = pd.DataFrame({"with_rerank": pivot_with, "without_rerank": pivot_without})
    improvement_df["improvement"] = improvement_df["with_rerank"] - improvement_df["without_rerank"]
    improvement_df["improvement_pct"] = improvement_df["improvement"] / improvement_df["without_rerank"] * 100
    improvement_df = improvement_df.reset_index()

    # Create visualization
    fig = px.bar(
        improvement_df,
        x="k",
        y="improvement_pct",
        color="rag_type",
        title=f"📈 Reranking Performance Improvement: {metric.replace('_', ' ').title()}",
        labels={"improvement_pct": "Improvement (%)", "k": "K (Retrieved Documents)"},
        barmode="group",
    )
    fig.update_xaxes(type='category')

    fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="No Improvement")
    fig.update_layout(height=400)

    return fig


# Plot reranking improvements
print("\n📊 RERANKING IMPROVEMENT VISUALIZATIONS")
print("=" * 50)

for metric in ["is_correct", 'is_correct_given_context', "retrieval_benchmarks_f1@10", 'retrieval_benchmarks_recall@10', "mrr"]:
    if metric in df.columns:
        fig = plot_reranking_delta(df, metric)
        if fig:
            fig.write_image(IMG_DIR / f'effect_of_reranking_{metric}.svg', height=500, width=800)
            fig.show()
    else:
        print(f"⚠️  Skipping {metric} visualization")


📊 RERANKING IMPROVEMENT VISUALIZATIONS


## 📊 Statistical Significance Testing

### 5. Hypothesis Testing for Key Research Questions

In [173]:
# Statistical Significance Testing
def test_reranking_significance(df, metric="is_correct", alpha=0.05):
    """Test if reranking significantly improves performance"""

    if metric not in df.columns:
        print(f"⚠️  Metric {metric} not available")
        return None

    test_results = []

    # Test for each k and RAG type combination
    for k_val in df["k"].unique():
        for rag_type in df["rag_type"].unique():
            # Get data subsets
            subset = df[(df["k"] == k_val) & (df["rag_type"] == rag_type)].dropna(subset=[metric])

            if len(subset) == 0:
                continue

            with_rerank = subset[subset["rerank"] == True][metric]
            without_rerank = subset[subset["rerank"] == False][metric]

            if len(with_rerank) > 0 and len(without_rerank) > 0:
                try:
                    # Use independent t-test since we don't have paired samples
                    t_stat, p_value = stats.ttest_ind(with_rerank, without_rerank)

                    # Calculate effect size (Cohen's d)
                    pooled_std = np.sqrt(
                        ((len(with_rerank) - 1) * with_rerank.var() + (len(without_rerank) - 1) * without_rerank.var())
                        / (len(with_rerank) + len(without_rerank) - 2)
                    )

                    if pooled_std > 0:
                        cohens_d = (with_rerank.mean() - without_rerank.mean()) / pooled_std
                    else:
                        cohens_d = 0

                    test_results.append(
                        {
                            "k": k_val,
                            "rag_type": rag_type,
                            "metric": metric,
                            "with_rerank_mean": with_rerank.mean(),
                            "without_rerank_mean": without_rerank.mean(),
                            "improvement": with_rerank.mean() - without_rerank.mean(),
                            "improvement_pct": (
                                (with_rerank.mean() - without_rerank.mean()) / without_rerank.mean() * 100
                            )
                            if without_rerank.mean() != 0
                            else 0,
                            "t_statistic": t_stat,
                            "p_value": p_value,
                            "significant": p_value < alpha,
                            "cohens_d": cohens_d,
                            "effect_size": "small"
                            if abs(cohens_d) < 0.5
                            else "medium"
                            if abs(cohens_d) < 0.8
                            else "large",
                            "n_with_rerank": len(with_rerank),
                            "n_without_rerank": len(without_rerank),
                        }
                    )
                except Exception as e:
                    print(f"⚠️  Error testing k={k_val}, rag_type={rag_type}: {e}")

    return pd.DataFrame(test_results)


# Run significance tests for key metrics
print("🔬 STATISTICAL SIGNIFICANCE TESTING")
print("=" * 60)

significance_results = {}
test_metrics = ["is_correct", "is_correct_given_context", "retrieval_benchmarks_f1@10", "mrr", "hit_rate"]

for metric in test_metrics:
    if metric in df.columns and df[metric].notna().any():
        print(f"\n📊 Testing significance for: {metric.replace('_', ' ').title()}")
        result = test_reranking_significance(df, metric)
        if result is not None and len(result) > 0:
            # Display summary
            sig_results = result[result["significant"] == True]
            print(f"Significant improvements: {len(sig_results)} out of {len(result)} tests")
            if len(sig_results) > 0:
                print("Most significant improvements:")
                print(
                    sig_results.nlargest(5, "improvement_pct")[
                        ["k", "rag_type", "improvement_pct", "p_value", "cohens_d", "effect_size"]
                    ]
                    .round(4)
                    .to_string(index=False)
                )

            significance_results[metric] = result
        else:
            print("⚠️  No valid test results")
    else:
        print(f"⚠️  Skipping {metric} - not available or all null")

🔬 STATISTICAL SIGNIFICANCE TESTING

📊 Testing significance for: Is Correct
Significant improvements: 2 out of 10 tests
Most significant improvements:
 k rag_type  improvement_pct  p_value  cohens_d effect_size
10    Basic          10.0156   0.0017    0.1770       small
10 Enhanced           6.5444   0.0134    0.1403       small

📊 Testing significance for: Is Correct Given Context
Significant improvements: 0 out of 10 tests

📊 Testing significance for: Retrieval Benchmarks F1@10
Significant improvements: 5 out of 6 tests
Most significant improvements:
 k rag_type  improvement_pct  p_value  cohens_d effect_size
50 Enhanced          29.3577   0.0000    0.4879       small
50    Basic          23.3577   0.0000    0.4172       small
10    Basic           9.5428   0.0002    0.1408       small
20 Enhanced           8.9054   0.0115    0.1504       small
10 Enhanced           5.7489   0.0111    0.0903       small

📊 Testing significance for: Mrr
Significant improvements: 7 out of 10 tests
Most 

In [174]:
# ================================================================================
# DETAILED RESULTS EXPORT AND SUMMARY
# ================================================================================

print("📋 EXPORTING DETAILED RESULTS")
print("=" * 50)

# Export enhanced results with timestamp
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

try:
    for metric, results_df in enhanced_significance_results.items():
        if results_df is not None and len(results_df) > 0:
            filename = f"enhanced_significance_{metric}_{timestamp}.csv"
            results_df.to_csv(filename, index=False)
            print(f"📁 Exported: {filename}")

            # Show key results summary
            print(f"\n📊 SUMMARY FOR {metric.upper()}:")
            print("-" * 40)

            # FDR corrected significant results
            sig_results = results_df[results_df["significant_fdr"] == True]
            print(f"Significant improvements (FDR): {len(sig_results)}/{len(results_df)}")

            if len(sig_results) > 0:
                # Best improvements
                best = sig_results.nlargest(3, "improvement_pct")
                print("\nTop 3 improvements:")
                for _, row in best.iterrows():
                    print(
                        f"  • k={row['k']}, {row['rag_type']}: {row['improvement_pct']:+.1f}% (d={row['cohens_d']:.3f})"
                    )

                # Average effect size for significant results
                avg_effect = sig_results["cohens_d"].mean()
                print(f"\nAverage effect size: {avg_effect:.3f}")

                # Effect size distribution
                effect_dist = sig_results["effect_size"].value_counts()
                print(f"Effect sizes: {dict(effect_dist)}")

            print()

except Exception as e:
    print(f"⚠️  Export error: {e}")

# ================================================================================
# FINAL COMPARATIVE SUMMARY
# ================================================================================

print("\n" + "=" * 80)
print("🎯 COMPARATIVE ANALYSIS: BASIC vs ENHANCED TESTING")
print("=" * 80)

if "significance_results" in locals() and "enhanced_significance_results" in locals():
    comparison_summary = []

    for metric in ["is_correct", "is_correct_given_context", "mrr", "hit_rate"]:
        if metric in significance_results and metric in enhanced_significance_results:
            # Basic results
            basic_df = significance_results[metric]
            basic_sig = len(basic_df[basic_df["significant"] == True]) if basic_df is not None else 0
            basic_total = len(basic_df) if basic_df is not None else 0

            # Enhanced results
            enhanced_df = enhanced_significance_results[metric]
            if enhanced_df is not None:
                enhanced_sig_raw = len(enhanced_df[enhanced_df["significant_raw"] == True])
                enhanced_sig_fdr = len(enhanced_df[enhanced_df["significant_fdr"] == True])
                enhanced_sig_bonf = len(enhanced_df[enhanced_df["significant_bonferroni"] == True])
                enhanced_total = len(enhanced_df)
            else:
                enhanced_sig_raw = enhanced_sig_fdr = enhanced_sig_bonf = enhanced_total = 0

            comparison_summary.append(
                {
                    "metric": metric,
                    "basic_significant": basic_sig,
                    "basic_total": basic_total,
                    "basic_rate": f"{basic_sig / basic_total * 100:.1f}%" if basic_total > 0 else "0%",
                    "enhanced_raw": enhanced_sig_raw,
                    "enhanced_fdr": enhanced_sig_fdr,
                    "enhanced_bonferroni": enhanced_sig_bonf,
                    "enhanced_total": enhanced_total,
                    "fdr_rate": f"{enhanced_sig_fdr / enhanced_total * 100:.1f}%" if enhanced_total > 0 else "0%",
                }
            )

    comparison_df = pd.DataFrame(comparison_summary)

    print("\n📊 SIGNIFICANCE RATES COMPARISON:")
    print("Metric                   | Basic  | Enhanced Raw | FDR Corrected | Bonferroni")
    print("-" * 75)
    for _, row in comparison_df.iterrows():
        metric_name = row["metric"].replace("_", " ").title()[:20].ljust(20)
        basic_rate = row["basic_rate"].rjust(6)
        raw_rate = f"{row['enhanced_raw']}/{row['enhanced_total']}".rjust(12)
        fdr_rate = f"{row['enhanced_fdr']}/{row['enhanced_total']}".rjust(13)
        bonf_rate = f"{row['enhanced_bonferroni']}/{row['enhanced_total']}".rjust(10)
        print(f"{metric_name} | {basic_rate} | {raw_rate} | {fdr_rate} | {bonf_rate}")

print("\n🔍 KEY INSIGHTS:")
print("• Basic testing likely shows inflated significance due to multiple comparisons")
print("• FDR correction provides balanced Type I/II error control")
print("• Bonferroni is most conservative, controls family-wise error rate")
print("• Enhanced testing includes effect sizes, power analysis, and robust methods")

print("\n✅ COMPLETE ENHANCED STATISTICAL ANALYSIS FINISHED!")
print("📁 All results exported with detailed methodology and corrections applied.")

📋 EXPORTING DETAILED RESULTS
📁 Exported: enhanced_significance_is_correct_20250920_224150.csv

📊 SUMMARY FOR IS_CORRECT:
----------------------------------------
Significant improvements (FDR): 1/10

Top 3 improvements:
  • k=10, Basic: +10.0% (d=0.177)

Average effect size: 0.177
Effect sizes: {'negligible': np.int64(1)}

📁 Exported: enhanced_significance_is_correct_given_context_20250920_224150.csv

📊 SUMMARY FOR IS_CORRECT_GIVEN_CONTEXT:
----------------------------------------
Significant improvements (FDR): 0/10

📁 Exported: enhanced_significance_retrieval_benchmarks_f1@10_20250920_224150.csv

📊 SUMMARY FOR RETRIEVAL_BENCHMARKS_F1@10:
----------------------------------------
Significant improvements (FDR): 5/6

Top 3 improvements:
  • k=50, Enhanced: +29.4% (d=0.494)
  • k=50, Basic: +23.4% (d=0.441)
  • k=10, Basic: +9.5% (d=0.141)

Average effect size: 0.263
Effect sizes: {'negligible': np.int64(3), 'small': np.int64(2)}

📁 Exported: enhanced_significance_mrr_20250920_224150.csv

In [175]:
# ================================================================================
# ENHANCED STATISTICAL SIGNIFICANCE TESTING WITH ROBUST METHODOLOGY
# ================================================================================

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest
import warnings

warnings.filterwarnings("ignore")


def check_assumptions(group1, group2):
    """Check statistical test assumptions"""
    from scipy.stats import shapiro, levene

    # Only test normality if sample size is reasonable (shapiro works best for n < 5000)
    if len(group1) <= 5000 and len(group2) <= 5000:
        _, p_norm1 = shapiro(group1)
        _, p_norm2 = shapiro(group2)
        normal = p_norm1 > 0.05 and p_norm2 > 0.05
    else:
        # For large samples, assume CLT applies
        normal = True

    # Equal variance test
    _, p_var = levene(group1, group2)
    equal_var = p_var > 0.05

    return {"normal": normal, "equal_var": equal_var}


def cohens_d_ci(group1, group2, confidence=0.95):
    """Calculate Cohen's d with confidence interval"""
    n1, n2 = len(group1), len(group2)

    # Pooled standard deviation
    s_pooled = np.sqrt(((n1 - 1) * group1.var() + (n2 - 1) * group2.var()) / (n1 + n2 - 2))

    # Cohen's d
    d = (group1.mean() - group2.mean()) / s_pooled if s_pooled > 0 else 0

    # Standard error for Cohen's d
    se_d = np.sqrt((n1 + n2) / (n1 * n2) + d**2 / (2 * (n1 + n2)))

    # Critical value
    from scipy.stats import t

    t_crit = t.ppf((1 + confidence) / 2, n1 + n2 - 2)

    # Confidence interval
    ci_low = d - t_crit * se_d
    ci_high = d + t_crit * se_d

    return ci_low, ci_high


def bootstrap_mean_diff(group1, group2, n_bootstrap=1000, confidence=0.95):
    """Bootstrap confidence interval for mean difference"""
    np.random.seed(42)  # For reproducibility
    diffs = []

    for _ in range(n_bootstrap):
        boot1 = np.random.choice(group1, len(group1), replace=True)
        boot2 = np.random.choice(group2, len(group2), replace=True)
        diffs.append(boot1.mean() - boot2.mean())

    alpha = 1 - confidence
    ci_low = np.percentile(diffs, 100 * alpha / 2)
    ci_high = np.percentile(diffs, 100 * (1 - alpha / 2))

    return ci_low, ci_high


def calculate_power(effect_size, n1, n2, alpha=0.05):
    """Calculate statistical power"""
    try:
        from statsmodels.stats.power import ttest_power

        return ttest_power(effect_size, n1 + n2, alpha, alternative="two-sided")
    except:
        # Fallback approximation
        return "Not calculated"


def enhanced_reranking_significance(df, metric="is_correct", alpha=0.05, min_practical_diff=0.05):
    """
    Enhanced significance testing with robust statistical methodology

    Improvements over basic approach:
    - Multiple testing correction (FDR)
    - Appropriate tests for binary vs continuous data
    - Assumption checking for parametric tests
    - Effect size confidence intervals
    - Bootstrap confidence intervals
    - Power analysis
    - Practical significance assessment
    """

    print(f"🔬 ENHANCED SIGNIFICANCE TESTING: {metric.replace('_', ' ').title()}")
    print("=" * 70)

    test_results = []
    all_p_values = []

    for k_val in sorted(df["k"].unique()):
        for rag_type in sorted(df["rag_type"].unique()):
            # Get data subsets
            subset = df[(df["k"] == k_val) & (df["rag_type"] == rag_type)].dropna(subset=[metric])

            if len(subset) == 0:
                continue

            with_rerank = subset[subset["rerank"] == True][metric]
            without_rerank = subset[subset["rerank"] == False][metric]

            # Require minimum sample size
            min_n = 5
            if len(with_rerank) < min_n or len(without_rerank) < min_n:
                continue

            try:
                # Determine if data is binary
                is_binary = set(with_rerank.unique()).issubset({0, 1, 0.0, 1.0}) and set(
                    without_rerank.unique()
                ).issubset({0, 1, 0.0, 1.0})

                # Select appropriate statistical test
                if is_binary:
                    # For binary data, use proportion test
                    count = [int(with_rerank.sum()), int(without_rerank.sum())]
                    nobs = [len(with_rerank), len(without_rerank)]
                    try:
                        z_stat, p_value = proportions_ztest(count, nobs)
                        test_type = "proportion_test"
                    except:
                        # Fallback to t-test
                        z_stat, p_value = stats.ttest_ind(with_rerank, without_rerank)
                        test_type = "t_test_fallback"
                else:
                    # For continuous data, check assumptions
                    assumptions = check_assumptions(with_rerank, without_rerank)

                    if assumptions["normal"] and assumptions["equal_var"]:
                        # Use independent t-test
                        z_stat, p_value = stats.ttest_ind(with_rerank, without_rerank)
                        test_type = "t_test"
                    else:
                        # Use Mann-Whitney U (non-parametric)
                        z_stat, p_value = stats.mannwhitneyu(with_rerank, without_rerank, alternative="two-sided")
                        test_type = "mann_whitney"

                # Calculate effect size (Cohen's d)
                pooled_std = np.sqrt((with_rerank.var() + without_rerank.var()) / 2)
                cohens_d = (with_rerank.mean() - without_rerank.mean()) / pooled_std if pooled_std > 0 else 0

                # Effect size confidence interval
                try:
                    d_ci_low, d_ci_high = cohens_d_ci(with_rerank, without_rerank)
                except:
                    d_ci_low, d_ci_high = np.nan, np.nan

                # Bootstrap confidence interval for mean difference
                try:
                    boot_ci_low, boot_ci_high = bootstrap_mean_diff(with_rerank, without_rerank)
                except:
                    boot_ci_low, boot_ci_high = np.nan, np.nan

                # Power analysis
                power = calculate_power(abs(cohens_d), len(with_rerank), len(without_rerank))

                # Calculate improvements
                mean_diff = with_rerank.mean() - without_rerank.mean()
                improvement_pct = (mean_diff / without_rerank.mean() * 100) if without_rerank.mean() != 0 else 0

                # Practical significance
                practically_significant = abs(improvement_pct) >= (min_practical_diff * 100)

                # Effect size interpretation
                if abs(cohens_d) < 0.2:
                    effect_size_interp = "negligible"
                elif abs(cohens_d) < 0.5:
                    effect_size_interp = "small"
                elif abs(cohens_d) < 0.8:
                    effect_size_interp = "medium"
                else:
                    effect_size_interp = "large"

                result = {
                    "k": k_val,
                    "rag_type": rag_type,
                    "metric": metric,
                    "test_type": test_type,
                    "with_rerank_mean": with_rerank.mean(),
                    "without_rerank_mean": without_rerank.mean(),
                    "mean_difference": mean_diff,
                    "improvement_pct": improvement_pct,
                    "test_statistic": z_stat,
                    "p_value_raw": p_value,
                    "cohens_d": cohens_d,
                    "cohens_d_ci_low": d_ci_low,
                    "cohens_d_ci_high": d_ci_high,
                    "bootstrap_ci_low": boot_ci_low,
                    "bootstrap_ci_high": boot_ci_high,
                    "effect_size": effect_size_interp,
                    "power": power,
                    "practically_significant": practically_significant,
                    "n_with_rerank": len(with_rerank),
                    "n_without_rerank": len(without_rerank),
                    "total_n": len(with_rerank) + len(without_rerank),
                }

                test_results.append(result)
                all_p_values.append(p_value)

            except Exception as e:
                print(f"⚠️  Error testing k={k_val}, rag_type={rag_type}: {e}")
                continue

    if not test_results:
        print("❌ No valid test results generated")
        return None

    # Convert to DataFrame
    results_df = pd.DataFrame(test_results)

    # Apply multiple testing correction
    if all_p_values:
        print(f"\n📊 Applying multiple testing correction to {len(all_p_values)} tests...")

        # FDR correction (Benjamini-Hochberg)
        rejected_fdr, p_corrected_fdr, _, _ = multipletests(all_p_values, method="fdr_bh")

        # Bonferroni correction (more conservative)
        rejected_bonf, p_corrected_bonf, _, _ = multipletests(all_p_values, method="bonferroni")

        # Add corrected p-values to results
        results_df["p_value_fdr"] = p_corrected_fdr
        results_df["significant_fdr"] = rejected_fdr
        results_df["p_value_bonferroni"] = p_corrected_bonf
        results_df["significant_bonferroni"] = rejected_bonf
        results_df["significant_raw"] = results_df["p_value_raw"] < alpha

        # Summary statistics
        raw_sig = sum(results_df["significant_raw"])
        fdr_sig = sum(results_df["significant_fdr"])
        bonf_sig = sum(results_df["significant_bonferroni"])

        print(f"\n📈 SIGNIFICANCE SUMMARY:")
        print(f"   Raw p < 0.05: {raw_sig}/{len(all_p_values)} tests ({raw_sig / len(all_p_values) * 100:.1f}%)")
        print(f"   FDR corrected: {fdr_sig}/{len(all_p_values)} tests ({fdr_sig / len(all_p_values) * 100:.1f}%)")
        print(
            f"   Bonferroni corrected: {bonf_sig}/{len(all_p_values)} tests ({bonf_sig / len(all_p_values) * 100:.1f}%)"
        )

        # Show most significant results (using FDR correction)
        significant_results = results_df[results_df["significant_fdr"] == True].copy()

        if len(significant_results) > 0:
            print(f"\n🎯 SIGNIFICANT IMPROVEMENTS (FDR corrected):")
            print("=" * 50)

            # Sort by effect size
            top_results = significant_results.nlargest(min(5, len(significant_results)), "cohens_d")

            for _, row in top_results.iterrows():
                print(f"\n📍 k={row['k']}, {row['rag_type']}:")
                print(f"   Improvement: {row['improvement_pct']:+.1f}% ({row['mean_difference']:+.4f})")
                print(f"   Effect size: {row['cohens_d']:.3f} ({row['effect_size']})")
                print(f"   p-value: {row['p_value_fdr']:.4f} (FDR corrected)")
                print(f"   Power: {row['power']}")
                print(f"   Sample sizes: {row['n_with_rerank']} vs {row['n_without_rerank']}")

            # Effect size distribution
            effect_counts = significant_results["effect_size"].value_counts()
            print(f"\n📊 Effect size distribution (significant results):")
            for effect, count in effect_counts.items():
                print(f"   {effect.title()}: {count}")

        else:
            print(f"\n❌ No statistically significant improvements found after correction")
            print("   This suggests either:")
            print("   - Reranking has no meaningful effect")
            print("   - Sample sizes are insufficient")
            print("   - Effect sizes are too small to detect reliably")

    return results_df


# Run enhanced significance testing
print("🚀 RUNNING ENHANCED STATISTICAL ANALYSIS")
print("=" * 80)

enhanced_significance_results = {}
test_metrics = ["is_correct", "is_correct_given_context", "retrieval_benchmarks_f1@10", "mrr", "hit_rate"]

for metric in test_metrics:
    if metric in df.columns and df[metric].notna().any():
        print(f"\n" + "=" * 80)
        result = enhanced_reranking_significance(df, metric, alpha=0.05, min_practical_diff=0.05)
        if result is not None:
            enhanced_significance_results[metric] = result
        print(f"\n✅ Completed analysis for {metric}")
    else:
        print(f"\n⚠️  Skipping {metric} - not available or all null")

print(f"\n\n🎊 ENHANCED STATISTICAL ANALYSIS COMPLETE!")
print("=" * 80)
print("\nKey improvements over basic analysis:")
print("• Multiple testing correction (FDR & Bonferroni)")
print("• Appropriate tests for binary vs continuous data")
print("• Effect size confidence intervals")
print("• Bootstrap confidence intervals")
print("• Power analysis")
print("• Practical significance assessment")
print("• Robust assumption checking")

🚀 RUNNING ENHANCED STATISTICAL ANALYSIS

🔬 ENHANCED SIGNIFICANCE TESTING: Is Correct

📊 Applying multiple testing correction to 10 tests...

📈 SIGNIFICANCE SUMMARY:
   Raw p < 0.05: 2/10 tests (20.0%)
   FDR corrected: 1/10 tests (10.0%)
   Bonferroni corrected: 1/10 tests (10.0%)

🎯 SIGNIFICANT IMPROVEMENTS (FDR corrected):

📍 k=10, Basic:
   Improvement: +10.0% (+0.0738)
   Effect size: 0.177 (negligible)
   p-value: 0.0169 (FDR corrected)
   Power: 0.9999925659012738
   Sample sizes: 639 vs 627

📊 Effect size distribution (significant results):
   Negligible: 1

✅ Completed analysis for is_correct

🔬 ENHANCED SIGNIFICANCE TESTING: Is Correct Given Context

📊 Applying multiple testing correction to 10 tests...

📈 SIGNIFICANCE SUMMARY:
   Raw p < 0.05: 0/10 tests (0.0%)
   FDR corrected: 0/10 tests (0.0%)
   Bonferroni corrected: 0/10 tests (0.0%)

❌ No statistically significant improvements found after correction
   This suggests either:
   - Reranking has no meaningful effect
   - S

### 6. Enhanced vs Basic RAG Comparison

In [176]:
# Enhanced vs Basic RAG Analysis
def analyze_enhancement_impact(df, metric="is_correct"):
    """Analyze the impact of RAG enhancement (query rewriting)"""

    if metric not in df.columns:
        print(f"⚠️  Metric {metric} not available")
        return None

    # Filter valid data
    analysis_df = df.dropna(subset=[metric])

    # Calculate mean performance for enhanced vs basic RAG
    enhancement_comparison = analysis_df.groupby(["k", "rerank_type", "rag_type"])[metric].mean().unstack()

    if "Enhanced" not in enhancement_comparison.columns or "Basic" not in enhancement_comparison.columns:
        print(f"⚠️  Missing enhancement comparison data for {metric}")
        return None

    # Calculate improvement from enhancement
    enhancement_comparison["enhancement_improvement"] = (
        enhancement_comparison["Enhanced"] - enhancement_comparison["Basic"]
    )
    enhancement_comparison["enhancement_improvement_pct"] = (
        enhancement_comparison["enhancement_improvement"] / enhancement_comparison["Basic"] * 100
    )

    return enhancement_comparison


# Analyze enhancement impact
print("🔧 ENHANCED vs BASIC RAG ANALYSIS")
print("=" * 50)

enhancement_results = {}
for metric in ["is_correct", "is_correct_given_context", "f1_at_1", "mrr"]:
    if metric in df.columns:
        print(f"\n📊 {metric.replace('_', ' ').title()} Enhancement Impact:")
        result = analyze_enhancement_impact(df, metric)
        if result is not None:
            print(result.round(4).to_string())
            enhancement_results[metric] = result
    else:
        print(f"⚠️  {metric} not available")

🔧 ENHANCED vs BASIC RAG ANALYSIS

📊 Is Correct Enhancement Impact:
rag_type            Basic  Enhanced  Full Context  enhancement_improvement  enhancement_improvement_pct
k  rerank_type                                                                                         
1  No Reranking    0.6063    0.6108        0.9259                   0.0045                       0.7439
   With Reranking  0.5803    0.6007           NaN                   0.0204                       3.5193
5  No Reranking    0.6910    0.7214        0.9206                   0.0304                       4.3985
   With Reranking  0.6704    0.7398           NaN                   0.0694                      10.3443
10 No Reranking    0.7368    0.7951        0.9171                   0.0583                       7.9094
   With Reranking  0.8106    0.8472           NaN                   0.0365                       4.5047
20 No Reranking    0.8414    0.8496        0.9232                   0.0081                       0.96

### 7. Interaction Effects & Advanced Visualizations

In [ ]:
# Interaction Effects Analysis
def create_interaction_plot(df, metric="is_correct"):
    """Create interaction plot showing how reranking benefit changes with RAG type and k"""

    if metric not in df.columns:
        return None

    # Calculate mean performance by all combinations
    interaction_df = df.dropna(subset=[metric]).groupby(["k", "rag_type", "rerank_type"])[metric].mean().reset_index()

    # Create interaction plot
    fig = px.line(
        interaction_df,
        x="k",
        y=metric,
        color="rag_type",
        line_dash="rerank_type",
        title=f"🔄 Interaction Effects: {metric.replace('_', ' ').title()}",
        labels={metric: f"{metric.replace('_', ' ').title()} Score", "k": "K (Retrieved Documents)"},
        markers=True,
    )

    fig.update_layout(
        height=500, xaxis_type="log", legend=dict(orientation="v", yanchor="top", y=1, xanchor="left", x=1.02)
    )

    return fig


# Box plots for distribution comparison
def create_distribution_comparison(df, metric="is_correct"):
    """Create box plots comparing distributions across configurations"""

    if metric not in df.columns:
        return None

    plot_df = df.dropna(subset=[metric])

    fig = px.box(
        plot_df,
        x="k",
        y=metric,
        color="config_label",
        title=f"📦 Distribution Comparison: {metric.replace('_', ' ').title()}",
        labels={metric: f"{metric.replace('_', ' ').title()} Score"},
    )

    fig.update_layout(height=500, xaxis_type="log")
    return fig


# Create interaction visualizations
print("🔄 INTERACTION EFFECTS ANALYSIS")
print("=" * 50)

interaction_metrics = ["is_correct", "f1_at_1", "mrr"]

for metric in interaction_metrics:
    if metric in df.columns:
        # Interaction plot
        fig = create_interaction_plot(df, metric)
        if fig:
            fig.show()

        # Distribution comparison
        fig2 = create_distribution_comparison(df, metric)
        if fig2:
            fig2.show()
    else:   
        print(f"⚠️  Skipping {metric} interaction analysis")

🔄 INTERACTION EFFECTS ANALYSIS


## 🎯 Key Findings & Recommendations

### 8. Summary of Analysis Results

In [178]:
# Generate Key Findings Summary
print("🎯 KEY FINDINGS SUMMARY")
print("=" * 60)

# 1. Overall performance comparison
print("\n1. OVERALL PERFORMANCE BY CONFIGURATION")
print("-" * 40)

# Calculate overall means for main metrics including context tokens
main_metrics = ["is_correct", "f1_at_1", "mrr", "avg_similarity", "context_tokens_used"]
config_performance = {}

for metric in main_metrics:
    if metric in df.columns and df[metric].notna().any():
        perf_by_config = df.groupby("config_label")[metric].agg(["mean", "std", "count"]).round(4)
        config_performance[metric] = perf_by_config
        print(f"\n📊 {metric.replace('_', ' ').title()}:")
        print(perf_by_config.to_string())

# 2. Best performing configurations
print(f"\n\n2. OPTIMAL K-VALUES BY CONFIGURATION")
print("-" * 40)

optimal_k = {}
for config in df["config_label"].unique():
    config_data = df[df["config_label"] == config]

    for metric in ["is_correct", "f1_at_1"]:
        if metric in config_data.columns and config_data[metric].notna().any():
            k_performance = config_data.groupby("k")[metric].mean()
            best_k = k_performance.idxmax()
            best_score = k_performance.max()

            key = f"{config}_{metric}"
            optimal_k[key] = {"k": best_k, "score": best_score}

            print(f"{config} - {metric}: k={best_k} (score={best_score:.4f})")

print(f"\n\n3. RERANKING IMPACT SUMMARY")
print("-" * 40)

# Summarize reranking benefits
if "reranking_results" in locals():
    for metric, results in reranking_results.items():
        if results is not None and "rerank_improvement_pct" in results.columns:
            positive_improvements = results[results["rerank_improvement_pct"] > 0]
            if len(positive_improvements) > 0:
                avg_improvement = positive_improvements["rerank_improvement_pct"].mean()
                max_improvement = positive_improvements["rerank_improvement_pct"].max()
                print(f"{metric}: Average improvement {avg_improvement:.2f}%, Max improvement {max_improvement:.2f}%")

print(f"\n\n4. CONTEXT EFFICIENCY ANALYSIS")
print("-" * 40)

if "context_tokens_used" in df.columns:
    # Token efficiency by configuration
    token_efficiency = df.groupby("config_label")[["context_tokens_used", "tokens_per_k"]].agg(["mean", "std"]).round(1)
    print("Context token usage by configuration:")
    print(token_efficiency.to_string())

    # Performance per token analysis
    if "is_correct" in df.columns:
        df_valid = df[(df["context_tokens_used"] > 0) & (df["is_correct"].notna())]
        if len(df_valid) > 0:
            df_valid["performance_per_token"] = df_valid["is_correct"] / (
                df_valid["context_tokens_used"] / 1000
            )  # per 1K tokens
            efficiency_by_config = (
                df_valid.groupby("config_label")["performance_per_token"].agg(["mean", "std"]).round(4)
            )
            print(f"\nPerformance per 1K context tokens:")
            print(efficiency_by_config.to_string())

print(f"\n\n5. STATISTICAL SIGNIFICANCE SUMMARY")
print("-" * 40)

if "significance_results" in locals():
    for metric, results in significance_results.items():
        if results is not None and len(results) > 0:
            significant_tests = results[results["significant"] == True]
            total_tests = len(results)
            significant_count = len(significant_tests)

            print(f"{metric}: {significant_count}/{total_tests} tests showed significant improvement")

            if len(significant_tests) > 0:
                large_effects = significant_tests[significant_tests["effect_size"] == "large"]
                medium_effects = significant_tests[significant_tests["effect_size"] == "medium"]
                print(f"  - Large effect sizes: {len(large_effects)}")
                print(f"  - Medium effect sizes: {len(medium_effects)}")

print(f"\n✅ Analysis complete! See executive summary at the top for main conclusions.")

🎯 KEY FINDINGS SUMMARY

1. OVERALL PERFORMANCE BY CONFIGURATION
----------------------------------------

📊 Is Correct:
                         mean     std  count
config_label                                
Basic RAG              0.7527  0.4315   3263
Basic RAG + Rerank     0.7746  0.4179   4974
Enhanced RAG           0.7965  0.4027   5434
Enhanced RAG + Rerank  0.7994  0.4005   4376
Full Context           0.9216  0.2688   6483

📊 F1 At 1:
                         mean     std  count
config_label                                
Basic RAG              0.3440  0.4751   7002
Basic RAG + Rerank     0.3897  0.4877   6027
Enhanced RAG           0.3076  0.4615   6027
Enhanced RAG + Rerank  0.4446  0.4970   6520
Full Context              NaN     NaN      0

📊 Mrr:
                         mean     std  count
config_label                                
Basic RAG              0.4266  0.4387   7002
Basic RAG + Rerank     0.4574  0.4545   6027
Enhanced RAG           0.3971  0.4269   6027
Enhan

In [ ]:
# TODO: After running the analysis above, update the executive summary at the top with:
# 1. Key findings on reranking effectiveness 
# 2. Optimal k-values for different configurations
# 3. Enhanced vs Basic RAG performance comparison
# 4. Statistical significance results
# 5. Production recommendations

from datetime import datetime

print("📝 Next Steps:")
print("1. Run all analysis cells above")
print("2. Review the results and statistical tests")
print("3. Update the executive summary at the top with concrete findings")
print("4. Add any additional shuffle variable analysis when the data is available")
print("5. Export results for reporting")

# Export key results to CSV with timestamps for further analysis
try:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Export configuration performance summary
    if 'config_performance' in locals():
        for metric, data in config_performance.items():
            filename = f"config_performance_{metric}_{timestamp}.csv"
            data.to_csv(filename)
            print(f"📁 Exported: {filename}")
    
    # Export significance test results  
    if 'significance_results' in locals():
        for metric, data in significance_results.items():
            filename = f"significance_tests_{metric}_{timestamp}.csv"
            data.to_csv(filename, index=False)
            print(f"📁 Exported: {filename}")
            
except Exception as e:
    print(f"⚠️  Export error: {e}")

print("✅ Notebook setup complete! Run all cells to generate the full analysis.")